# Phase 3 qubit-scaling diagnostic

Tiny bounded test for the question: is the RF-QRC failure mainly a 6-qubit capacity bottleneck?

This does **not** run a full broad sweep. It checks whether q95 transition/event classification improves as the same RF-QRC ring map scales from 6 to 8 to 10 qubits.

Models compared:

```text
HAR memory classifier
QRC ring n-qubit classifier
HAR + QRC ring n-qubit classifier
```

Default subset sizes are intentionally bounded so this can run overnight if needed.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 240)

def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if p.name == 'qpitome-qrc-volatility' and (p / 'scripts').exists() and (p / 'src').exists():
            return p
    raise RuntimeError('Open this notebook from inside qpitome-qrc-volatility.')

ROOT = find_repo_root()
os.chdir(ROOT)
TABLES = ROOT / 'results' / 'tables'
FIGURES = ROOT / 'results' / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
print('Repo root:', ROOT)


## Run the bounded scaling diagnostic

Start with 6/8/10 qubits. If 10 qubits is too slow, interrupt and rerun with only 6/8. If it runs comfortably, 12 can be tried later manually.

In [ ]:
script = ROOT / 'scripts' / 'run_phase3_qubit_scaling_diagnostic.py'
if not script.exists():
    raise FileNotFoundError(script)

cmd = [
    sys.executable, str(script),
    '--n-qubits', '6', '8', '10',
    '--input-mode', 'level_rate',
    '--zz-mode', 'ring',
    '--max-train', '1800',
    '--max-val', '600',
    '--max-test', '800',
    '--leak', '0.3',
    '--logistic-C', '1.0',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)


## Load results

In [ ]:
metrics_path = TABLES / 'phase3_qubit_scaling_transition_classifier_metrics.csv'
test_path = TABLES / 'phase3_qubit_scaling_transition_classifier_test_summary.csv'
pred_path = TABLES / 'phase3_qubit_scaling_transition_classifier_predictions.csv'

metrics = pd.read_csv(metrics_path)
test = pd.read_csv(test_path)
preds = pd.read_csv(pred_path)

display(test.sort_values(['event', 'f1'], ascending=[True, False]))
display(preds.head())


## q95 focus

In [ ]:
cols = [
    'model', 'event', 'n_qubits', 'features', 'effective_rank',
    'precision', 'recall', 'f1', 'average_precision', 'roc_auc',
    'called_rate', 'event_rate', 'threshold', 'elapsed_feature_seconds',
]
q95 = test[test['event'].eq('q95')].copy()
q95_path = TABLES / 'phase3_qubit_scaling_q95_focus.csv'
q95.to_csv(q95_path, index=False)
print('Saved:', q95_path)
display(q95[cols].sort_values('f1', ascending=False))


## Figures

In [ ]:
def savefig(path: Path):
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches='tight')
    print('Saved:', path)
    plt.show()

# QRC-only scaling curves.
qrc = test[test['model'].str.startswith('QRC_ring_')].copy()
for metric in ['f1', 'average_precision', 'roc_auc', 'effective_rank']:
    fig, ax = plt.subplots(figsize=(8, 5))
    for event, g in qrc.groupby('event'):
        g = g.sort_values('n_qubits')
        ax.plot(g['n_qubits'], g[metric], marker='o', label=event)
    ax.set_xlabel('n qubits')
    ax.set_ylabel(metric)
    ax.set_title(f'QRC ring scaling: {metric}')
    ax.legend()
    savefig(FIGURES / f'phase3_qubit_scaling_qrc_{metric}.png')

# q95 model comparison.
plot = q95.sort_values('f1', ascending=False).copy()
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(plot['model'], plot['f1'])
ax.set_ylabel('q95 F1')
ax.set_title('q95 transition/event classifier comparison')
ax.tick_params(axis='x', rotation=35)
savefig(FIGURES / 'phase3_qubit_scaling_q95_f1_model_comparison.png')

fig, ax = plt.subplots(figsize=(7, 5))
for model, g in q95.groupby('model'):
    ax.scatter(g['average_precision'], g['f1'], label=model)
ax.set_xlabel('average precision')
ax.set_ylabel('q95 F1')
ax.set_title('q95 AP vs thresholded F1')
ax.legend(fontsize=7)
savefig(FIGURES / 'phase3_qubit_scaling_q95_ap_vs_f1.png')


## Decision rule

In [ ]:
har = q95[q95['model'].eq('HAR_memory_classifier')].iloc[0]
best_qrc = q95[q95['model'].str.startswith('QRC_ring_')].sort_values('f1', ascending=False).iloc[0]
best_hq = q95[q95['model'].str.startswith('HAR_plus_QRC')].sort_values('f1', ascending=False).iloc[0]

print('HAR q95 F1:', float(har['f1']))
print('Best QRC-only q95 F1:', best_qrc['model'], float(best_qrc['f1']))
print('Best HAR+QRC q95 F1:', best_hq['model'], float(best_hq['f1']))

if float(best_hq['f1']) > float(har['f1']):
    print('HAR+QRC improves q95 classification over HAR on this bounded test.')
elif float(best_qrc['f1']) > float(har['f1']):
    print('QRC-only improves over HAR, but concatenation does not. Inspect calibration/thresholding.')
else:
    print('HAR still dominates q95 classification on this bounded test. Larger qubits did not rescue the signal here.')
